# Dependencies 
`mvr_regex` needs the optional [`greenery`](https://pypi.org/project/greenery/)
package (`pip install conin[regex]`); without it the call raises `ImportError`.

# Building MVR constraints from regular expressions

`mvr_regex(hidden_markov_model, pattern)` compiles a regular expression over
hidden-state *labels* into a `HomMVR`: the pattern is turned into a minimal DFA,
whose states become the mediation states. The MVR evaluates to `True` at time
`t` exactly when the hidden states consumed so far, `h_0 … h_t`, match the
pattern **in full** — there is no partial or substring match, and no anchors.

The syntax is summarised in [`REGEX_CHEATSHEET.md`](REGEX_CHEATSHEET.md), which also carries the error messages, the range convention and how a pattern behaves under a `time_range`.

In [ ]:
from itertools import product

from conin.constraint import mvr_constraint_fn
from conin.exceptions import InvalidInputError
from conin.hidden_markov_model import ConstrainedHiddenMarkovModel
from conin.hidden_markov_model.hmm import HiddenMarkovModel
from conin.hidden_markov_model.mvr_constraints import mvr_holdingtime, mvr_regex

In [ ]:
def make_hmm(hidden_states, observed_states=("quiet", "loud")):
    """A uniform HMM over the given labels; only the labels matter here."""
    hidden_states, observed_states = list(hidden_states), list(observed_states)
    hmm = HiddenMarkovModel()
    hmm.load_model(
        start_probs={h: 1 / len(hidden_states) for h in hidden_states},
        transition_probs={
            (a, b): 1 / len(hidden_states)
            for a in hidden_states
            for b in hidden_states
        },
        emission_probs={
            (h, o): 1 / len(observed_states)
            for h in hidden_states
            for o in observed_states
        },
        initialize=True,
    )
    return hmm


def accepts(mvr, seq):
    """Run a HomMVR over a nonempty hidden sequence and return its verdict."""
    m = mvr.ini[seq[0]]
    for h in seq[1:]:
        m = mvr.upd[(m, h)]
    return mvr.evl[m]

## Regex Syntax

A hidden state is always written `<label>`, using the label's `str()` form.
Everything outside the angle brackets is regular-expression syntax; a bare
character is an error rather than a literal, so a pattern can never silently
mean something other than what it reads as.

| Syntax | Meaning | Example |
| --- | --- | --- |
| `<label>` | the hidden state `label` | `<a>` |
| `.` | any hidden state | `<a>.<c>` |
| `\|` | alternation | `<a>\|<b><c>` |
| `( … )` | grouping | `(<a><b>)*<c>` |
| `*` `+` `?` | zero-or-more, one-or-more, optional | `<a><b>*<c>` |
| `{m}` `{m,n}` `{m,}` | bounded repetition | `<a>{2,3}` |
| `[ … ]` | one of a set of states | `[<a><b>]<c>` |
| `[^ … ]` | any state except | `[^<a>]<c>` |
| `[<a>-<c>]` | a range, in the model's hidden-state order | `[<a>-<b>]<c>` |

Not supported: anchors (`^`, `$`), backslash escapes and character classes such
as `\d`, and backreferences. Anchors are rejected with their own message
because a pattern already matches the whole sequence.

## Example 1 — one `a`, then any number of `b`, then `c`

The pattern below accepts `ac`, `abc`, `abbc`, …

In [ ]:
hmm = make_hmm(["a", "b", "c"])
mvr = mvr_regex(hmm, "<a><b>*<c>")

[
    "".join(seq)
    for length in range(1, 5)
    for seq in product("abc", repeat=length)
    if accepts(mvr, list(seq))
]

The mediation states are the DFA's states, using `greenery`'s integer names:

- `1` is "seen an `a`, waiting for `c`"
- `2` is the absorbing failure state
- `3` is "matched".

`mvr_regex` prunes unreachable states before returning, so the mediation space is already minimal.

In [ ]:
print("mediation_states:", mvr.mediation_states)
print("ini:", mvr.ini)
print("upd:", mvr.upd)
print("evl:", mvr.evl)

## Example 2 — Washing Machine's Wash Cycle

A washing machine passes through `fill`, `wash`, `rinse` and `spin` in that
order, spending at least one step in each. That is one pattern, and it is the
kind of ordering condition the constraint primitives could only express by
composition.

In [ ]:
machine = make_hmm(["fill", "wash", "rinse", "spin"])
cycle = mvr_regex(machine, "<fill>+<wash>+<rinse>+<spin>+")

traces = [
    ["fill", "wash", "rinse", "spin"],
    ["fill", "fill", "wash", "wash", "wash", "rinse", "spin", "spin"],
    ["fill", "wash", "spin"],
    ["wash", "rinse", "spin"],
    ["fill", "wash", "rinse", "spin", "wash"],
]

for trace in traces:
    print(f"{accepts(cycle, trace)!s:>5}  {' '.join(trace)}")

Wrapping it in `@mvr_constraint_fn` gives a constraint that a
`ConstrainedHiddenMarkovModel` builds at `initialize_chmm` time.

In [ ]:
@mvr_constraint_fn(name="wash_cycle")
def wash_cycle(hidden_markov_model):
    return mvr_regex(hidden_markov_model, "<fill>+<wash>+<rinse>+<spin>+")


chmm = ConstrainedHiddenMarkovModel(hmm=machine, constraints=[wash_cycle])
chmm.initialize_chmm()

chmm.chmm.constraints[0].mediation_states

## A regex is not always the shortest route

`mvr_regex` is convenient, not automatically preferable. Where a dedicated
constructor exists it is usually smaller and its conventions are stated, while a
pattern that looks equivalent may not be.

Compare "every run of `a` lasts at least 3 steps" written both ways. They
disagree on every sequence ending mid-`a`-run: `mvr_holdingtime` **exempts the
final run**, which may continue past the horizon, whereas the pattern has no
such notion and simply fails.

In [ ]:
hmm = make_hmm(["a", "b", "c"])
pattern_version = mvr_regex(hmm, "(<a>{3,}|<b>|<c>)*")
holdingtime_version = mvr_holdingtime(hmm, 3, states={"a"})

disagree = [
    "".join(seq)
    for length in range(1, 5)
    for seq in product("abc", repeat=length)
    if accepts(pattern_version, list(seq)) != accepts(holdingtime_version, list(seq))
]

print(f"{len(disagree)} sequences differ, all ending mid-run:", disagree[:6], "...")

## Errors

Every rejection names the offending index, so a malformed pattern fails at
construction rather than quietly matching something else.

In [ ]:
for pattern in ["<a>b", "^<a>", "<a", "<z>", r"\d"]:
    try:
        mvr_regex(hmm, pattern)
    except InvalidInputError as error:
        print(f"{pattern!r:>6}  {error}")

A pattern that is *valid* but matches nothing an MVR can see — including the
empty pattern, since an MVR always consumes at least one hidden state — is built
anyway, as a constantly-false constraint, with a warning.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    mvr_regex(hmm, "")

print(caught[0].message)